# CubifyAnything + Qwen3.5-9B: inferência, classificação dos crops e re-render

Pipeline end-to-end pensado para **Google Colab com GPU A100**:

1. Clona o repo `ml-cubifyanything` e instala dependências.
2. Roda o `CutrRunner` (mesma lógica do `tools/infer_image.py`) na imagem.
3. Extrai crops 2D de cada detecção (em memória; opcionalmente salva em disco).
4. Carrega o **Qwen3.5-9B** (bf16) uma única vez.
5. Para cada crop, pede ao Qwen uma linha `Image Tags: ...` e usa a **primeira tag** como `category` da detecção, gravando também a lista completa em `qwen_tags` e a linha bruta em `qwen_raw`.
6. Salva o `*_inf.json` atualizado (mesmo schema do `tools/infer_image.py`) e re-renderiza o `*_inf.png` com as categorias do Qwen via `tools/visualize_preds.draw_predictions`.

**Inputs esperados** (suba para a sessão do Colab antes de rodar):

- Uma imagem RGB (ex.: `teste/teste_final.png`)
- O JSON de passthrough com intrínsecos (ex.: `teste/teste_final.json`) — opcional
- O checkpoint CuTR (ex.: `models/cutr_rgb.pth`)

> **Runtime**: `Runtime → Change runtime type → A100 GPU`. Qwen 9B em bf16 ocupa ~18 GB, então T4 (16 GB) **não** é suportado neste notebook.

## 1. Setup — clonar repo e instalar dependências

Rode apenas uma vez por sessão do Colab. Se o repo já estiver clonado (re-execução da célula) o `git clone` é pulado.

In [ ]:
import os, sys, subprocess, pathlib

REPO_DIR = "/content/ml-cubifyanything"

if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(
        ["git", "clone", "https://github.com/apple/ml-cubifyanything.git", REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

# Instala só o necessário (sem cyclonedds/rerun-sdk/webdataset, que são para a demo CLI).
%pip install -q -e .
%pip install -q timm scipy tifffile Pillow numpy
%pip install -q "transformers>=4.45" accelerate

# Permite importar tools/ como pacotes simples (mesmo truque que tools/infer_image.py faz).
sys.path.insert(0, str(pathlib.Path(REPO_DIR) / "tools"))
print("sys.path[0] =", sys.path[0])

## 2. Configuração

Ajuste os caminhos para apontar para os arquivos que você subiu na sessão (use o painel lateral do Colab ou monte o Google Drive). Os defaults reproduzem os dois comandos originais:

```bash
python tools/infer_image.py --image teste/teste_final.png --meta-json teste/teste_final.json \
    --model-path models/cutr_rgb.pth --device cuda --max-edge 0 --score-thresh 0.35
python tools/visualize_preds.py --image teste/teste_final.png --pred-json teste/teste_final_inf.json \
    --crops-dir teste/teste_final_crops
```

In [ ]:
from pathlib import Path

IMAGE_PATH    = Path("teste/teste_final.png")
META_JSON     = Path("teste/teste_final.json")   # use None se não tiver
MODEL_PATH    = Path("models/cutr_rgb.pth")
DEVICE        = "cuda"
MAX_EDGE      = 0                                  # 0 = sem resize
SCORE_THRESH  = 0.35                               # filtro aplicado ANTES do Qwen
QWEN_MODEL_ID = "Qwen/Qwen3.5-9B"
SAVE_CROPS    = False                              # True replica o tools/visualize_preds.py --crops-dir
OUT_JSON      = None                               # default: <image>_inf.json
OUT_PNG       = None                               # default: <image>_inf.png

assert IMAGE_PATH.exists(), f"Imagem não encontrada: {IMAGE_PATH}"
assert MODEL_PATH.exists(), f"Checkpoint CuTR não encontrado: {MODEL_PATH}"
if META_JSON is not None and not META_JSON.exists():
    print(f"WARN: meta JSON {META_JSON} não existe — seguindo sem intrínsecos.")
    META_JSON = None

image_stem = "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in IMAGE_PATH.stem)
if OUT_JSON is None:
    OUT_JSON = IMAGE_PATH.with_name(image_stem + "_inf.json")
if OUT_PNG is None:
    OUT_PNG  = IMAGE_PATH.with_name(image_stem + "_inf.png")
CROPS_DIR = IMAGE_PATH.with_name(image_stem + "_crops") if SAVE_CROPS else None

print("IMAGE_PATH :", IMAGE_PATH)
print("META_JSON  :", META_JSON)
print("MODEL_PATH :", MODEL_PATH)
print("OUT_JSON   :", OUT_JSON)
print("OUT_PNG    :", OUT_PNG)
print("CROPS_DIR  :", CROPS_DIR)

## 3. Inferência CuTR

Importa direto de `tools/cutr_runtime.py` e `tools/infer_image.py` (já estão no `sys.path`). O bloco abaixo reproduz a lógica de `tools/infer_image.py` — `load_meta_json`, montagem de `K_user`, `CutrRunner.infer` — sem chamar o script via subprocess.

In [ ]:
import os, sys, pathlib
import torch
from PIL import Image

# Belt-and-suspenders: garante cwd no repo e tools/ no sys.path mesmo
# após restart do runtime (o %pip da célula 1 às vezes força isso no Colab).
REPO_DIR  = pathlib.Path("/content/ml-cubifyanything")
TOOLS_DIR = REPO_DIR / "tools"
if pathlib.Path.cwd() != REPO_DIR and REPO_DIR.exists():
    os.chdir(REPO_DIR)
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from cutr_runtime import CutrRunner, make_default_intrinsics
from infer_image import load_meta_json, _meta_intrinsic, save_pred_json

meta = load_meta_json(META_JSON) if META_JSON is not None else None

runner = CutrRunner(model_path=str(MODEL_PATH), device=DEVICE)
img = Image.open(str(IMAGE_PATH)).convert("RGB")
w_orig, h_orig = img.size

fx = _meta_intrinsic(meta, "fx")
fy = _meta_intrinsic(meta, "fy")
cx = _meta_intrinsic(meta, "cx")
cy = _meta_intrinsic(meta, "cy")

K_user = None
if any(v is not None for v in (fx, fy, cx, cy)):
    K_user = make_default_intrinsics(w_orig, h_orig)
    if fx is not None: K_user[0, 0] = float(fx)
    if fy is not None: K_user[1, 1] = float(fy)
    if cx is not None: K_user[0, 2] = float(cx)
    if cy is not None: K_user[1, 2] = float(cy)
    print(f"Intrinsics (meta): fx={fx}, fy={fy}, cx={cx}, cy={cy} (image {w_orig}x{h_orig})")

max_edge = None if (MAX_EDGE is None or int(MAX_EDGE) <= 0) else int(MAX_EDGE)

pred = runner.infer(
    image=img,
    K=K_user,
    depth_m=None,
    score_thresh=float(SCORE_THRESH),
    max_edge=max_edge,
)

print(f"Detecções após score_thresh={SCORE_THRESH}: {len(pred.get('detections', []))}")

## 4. Extração dos crops

Mesmo loop e regras de `tools/visualize_preds.py` (filtra por score, clampa bbox, descarta bbox < 2 px). Os crops ficam em memória como `(i, PIL.Image)` para reconectar a tag do Qwen à detecção `pred['detections'][i]`. Com `SAVE_CROPS=True`, também salva em disco como `crop_NNN.png` (compatível com o output atual).

In [ ]:
def _clamp(v, lo, hi):
    return max(lo, min(hi, v))

if SAVE_CROPS:
    CROPS_DIR.mkdir(parents=True, exist_ok=True)

w, h = img.size
dets = pred.get("detections", []) or []
crops = []  # list of (index_in_detections, PIL.Image)

for i, det in enumerate(dets):
    score = det.get("score")
    if score is not None and float(score) < float(SCORE_THRESH):
        continue
    bbox = det.get("bbox_xyxy")
    if not bbox or len(bbox) != 4:
        continue
    x1, y1, x2, y2 = [float(v) for v in bbox]
    x1 = _clamp(x1, 0.0, w - 1.0); y1 = _clamp(y1, 0.0, h - 1.0)
    x2 = _clamp(x2, 0.0, w - 1.0); y2 = _clamp(y2, 0.0, h - 1.0)
    x1_i = max(0, int(round(x1))); y1_i = max(0, int(round(y1)))
    x2_i = max(0, int(round(x2))); y2_i = max(0, int(round(y2)))
    if x2_i - x1_i < 2 or y2_i - y1_i < 2:
        continue
    crop = img.crop((x1_i, y1_i, x2_i, y2_i))
    crops.append((i, crop))
    if SAVE_CROPS:
        crop.save(CROPS_DIR / f"crop_{i:03d}.png")

print(f"Crops aceitos: {len(crops)} (de {len(dets)} detecções)")
if SAVE_CROPS:
    print(f"Crops salvos em: {CROPS_DIR}")

## 5. Carregar o Qwen3.5-9B (bf16, A100)

Carregamento único — o modelo fica em memória pelas próximas células. O `trust_remote_code=True` é necessário para o template de chat do Qwen.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Qwen carregado em", next(model.parameters()).device)

## 6. Helpers do Qwen

Mesmo prompt e parsing do código que você passou: extrai a linha `Image Tags: ...` e quebra por `|`.

In [ ]:
import re

OPEN_TAG_PROMPT = """Você é um classificador de imagens (image tagging).

A imagem é um RECORTE (crop). Liste apenas o que está claramente visível dentro do recorte.

Regras:
- Português, substantivos curtos (1–4 palavras).
- Não invente. Em dúvida, omita.
- Sem duplicatas. Máximo 20 itens.

FORMATO DE SAÍDA (obrigatório):
Responda com UMA ÚNICA linha começando exatamente com:
Image Tags:
seguido de tags separadas por " | " (espaço-barra-espaço).

Exemplo:
Image Tags: caneta | mesa | papel | madeira

Não use JSON. Não use markdown. Não explique. Não mostre raciocínio."""


def extract_image_tags_line(texto: str):
    if not texto:
        return None
    m = re.search(r"(Image Tags:\s*.+)", texto, flags=re.IGNORECASE | re.DOTALL)
    if not m:
        return None
    return m.group(1).split("\n")[0].strip()


def parse_tags_from_response(texto: str):
    line = extract_image_tags_line(texto)
    if not line:
        return []
    body = re.sub(r"^Image Tags:\s*", "", line, flags=re.IGNORECASE).strip()
    return [t.strip() for t in body.split("|") if t.strip()]


def tag_crop(pil_image, max_new_tokens: int = 256):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": OPEN_TAG_PROMPT},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        chat_template_kwargs={"enable_thinking": False},
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
        )
    trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated)
    ]
    texto = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    return texto, parse_tags_from_response(texto), extract_image_tags_line(texto)

## 7. Classificar cada crop e injetar no JSON

A primeira tag vira `det['category']` (campo que o `tools/visualize_preds.py` já lê). A lista completa fica em `det['qwen_tags']` e a linha bruta em `det['qwen_raw']`.

In [ ]:
for i, pil in crops:
    _, tags, tags_line = tag_crop(pil)
    det = pred["detections"][i]
    det["category"]  = tags[0] if tags else None
    det["qwen_tags"] = tags
    det["qwen_raw"]  = tags_line
    print(f"[{i:03d}] {tags_line or '(sem tag)'}")

n_cat = sum(1 for d in pred["detections"] if d.get("category"))
print(f"\nClassificação concluída: {n_cat}/{len(pred['detections'])} detecções com categoria do Qwen.")

## 8. Salvar `*_inf.json` atualizado

Reusa `save_pred_json` de `tools/infer_image.py` — mesmo schema (`source_image`, `timestamp`, `video_id`, `image_size_hw`, `detections`, etc.), só com os campos extras `category` / `qwen_tags` / `qwen_raw` por detecção. O `tools/visualize_preds.py` original continua funcionando com este JSON.

In [ ]:
save_pred_json(pred, image_path=IMAGE_PATH, out_path=OUT_JSON)
print(f"JSON salvo: {OUT_JSON}")

## 9. Re-renderizar `*_inf.png` com as categorias do Qwen

Usa `draw_predictions` de `tools/visualize_preds.py` com `category_from="category"`, que já consome o campo que acabamos de gravar.

In [ ]:
import sys, pathlib
TOOLS_DIR = pathlib.Path('/content/ml-cubifyanything/tools')
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from IPython.display import display
from visualize_preds import draw_predictions

out_img = draw_predictions(
    img,
    pred,
    score_thresh=float(SCORE_THRESH),
    show_labels=True,
    line_width=3,
    category_from="category",
)
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
out_img.save(str(OUT_PNG))
print(f"Imagem salva: {OUT_PNG}")
display(out_img)

## 10. Baixar os artefatos (opcional)

Use o painel de arquivos do Colab (botão direito → *Download*) ou descomente a célula abaixo para baixar via `google.colab.files`.

In [ ]:
# from google.colab import files
# files.download(str(OUT_JSON))
# files.download(str(OUT_PNG))